# 🎓 Rexell Resale-Price LSTM Model Training

Welcome to the interactive training notebook for the Rexell AI forecasting engine. This notebook parses transaction logs to:
1. **Generate Data-Driven Heuristics** (`event_stats.json` for fallback and demand prediction).
2. **Calculate Scale Modifiers** (`resale_scaler.json` for min/max value normalization).
3. **Train a Recurrent Neural Network (LSTM)** in Keras to predict future transaction markup ratios (`resale_lstm.keras`).

### ⚠️ Environment Compatibility Note
- If you are running **Python 3.14.0+ on Windows**, pre-compiled TensorFlow binaries are not yet available on PyPI.
- **Recommended Action:** Click the button below to open and run this notebook in **Google Colab** (which runs Python 3.10 with GPU acceleration and pre-installed TensorFlow/Keras packages):

<a href="https://colab.research.google.com/" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install & Import dependencies
!pip install tensorflow numpy pandas matplotlib

import os
import csv
import json
import re
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import numpy as np
import tensorflow as tf

ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)
ERROR: No matching distribution found for tensorflow


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# 2. Self-contained features & sequence processing logic

WINDOW = 8  # sliding window size
_FALLBACK_RATIO = 1.0

def _parse_ts(value: str):
    value = (value or "").strip()
    for fmt in ("%Y-%m-%d %H:%M:%S.%f", "%Y-%m-%d %H:%M:%S", "%Y-%m-%d"):
        try:
            return datetime.strptime(value, fmt)
        except ValueError:
            continue
    return None

def _to_float(value: str, default: float = 0.0) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return default

def load_rows(csv_path: str):
    rows = []
    with open(csv_path, newline="") as fh:
        reader = csv.DictReader(fh)
        for r in reader:
            ts = _parse_ts(r.get("timestamp", ""))
            price = _to_float(r.get("price_paid"))
            original = _to_float(r.get("original_event_price"))
            if original <= 0:
                continue
            rows.append({
                "event_id": (r.get("event_id") or "").strip(),
                "timestamp": ts,
                "price_paid": price,
                "original_event_price": original,
                "ticket_count": _to_float(r.get("ticket_count"), 1.0),
                "markup_ratio": price / original if original else _FALLBACK_RATIO,
                "is_resale": (r.get("is_resale") or "").strip().lower() == "true",
            })
    rows.sort(key=lambda x: (x["timestamp"] or datetime.min))
    return rows

def build_sequences(rows, window: int = WINDOW):
    ratios = [r["markup_ratio"] for r in rows if r["markup_ratio"] > 0]
    X = []
    y = []
    for i in range(len(ratios) - window):
        X.append(ratios[i : i + window])
        y.append(ratios[i + window])
    return X, y

def build_event_stats(rows, window: int = WINDOW):
    by_event = defaultdict(list)
    for r in rows:
        if r["event_id"]:
            by_event[r["event_id"]].append(r)
    stats = {}
    for event_id, evrows in by_event.items():
        ratios = sorted(r["markup_ratio"] for r in evrows if r["markup_ratio"] > 0)
        resale_ratios = sorted(r["markup_ratio"] for r in evrows if r["is_resale"] and r["markup_ratio"] > 0)
        days = {r["timestamp"].date() for r in evrows if r["timestamp"]}
        n_days = max(len(days), 1)
        total_tickets = sum(r["ticket_count"] for r in evrows)
        last_window = [r["markup_ratio"] for r in evrows[-window:] if r["markup_ratio"] > 0]
        stats[event_id] = {
            "count": len(evrows),
            "median_markup": _median(ratios),
            "median_resale_markup": _median(resale_ratios) if resale_ratios else _median(ratios),
            "avg_daily_demand": round(total_tickets / n_days, 3),
            "last_window": last_window,
        }
    return stats

def global_fallback(rows):
    ratios = sorted(r["markup_ratio"] for r in rows if r["markup_ratio"] > 0)
    resale = sorted(r["markup_ratio"] for r in rows if r["is_resale"] and r["markup_ratio"] > 0)
    total_tickets = sum(r["ticket_count"] for r in rows)
    days = {r["timestamp"].date() for r in rows if r["timestamp"]}
    return {
        "median_markup": _median(ratios) if ratios else _FALLBACK_RATIO,
        "median_resale_markup": _median(resale) if resale else (_median(ratios) if ratios else _FALLBACK_RATIO),
        "avg_daily_demand": round(total_tickets / max(len(days), 1), 3),
    }

def _median(values):
    if not values:
        return _FALLBACK_RATIO
    values = sorted(values)
    n = len(values)
    mid = n // 2
    if n % 2:
        return values[mid]
    return (values[mid - 1] + values[mid]) / 2.0

In [ ]:
# 3. Load Dataset & Generate Event Statistics
# Note: If running on Colab, upload 'blockchain_ticketing_master.csv' using the Files tab on the left!

CSV_PATH = "../../../../dataset/blockchain_ticketing_master.csv"
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not Path(CSV_PATH).exists():
    # Fallback default location check
    CSV_PATH = "blockchain_ticketing_master.csv"
    if not Path(CSV_PATH).exists():
        raise FileNotFoundError("Please upload 'blockchain_ticketing_master.csv' to continue.")

print(f"Loading transactions from {CSV_PATH}")
rows = load_rows(CSV_PATH)
print(f"Loaded {len(rows)} transactions successfully!")

# Build statistics
event_stats = build_event_stats(rows)
global_stats = global_fallback(rows)

stats_path = MODEL_DIR / "event_stats.json"
with open(stats_path, "w") as fh:
    json.dump({"events": event_stats, "global": global_stats}, fh, indent=2)

print(f"Saved baseline metrics to: {stats_path}")

In [ ]:
# 4. Prepare Sequences & Build Scale limits
X, y = build_sequences(rows)
if len(X) < 10:
    raise ValueError(f"Not enough sequence data (found {len(X)} sequences). Add more transactions to the CSV dataset.")

X_arr = np.array(X, dtype="float32")
y_arr = np.array(y, dtype="float32")

lo = float(min(X_arr.min(), y_arr.min()))
hi = float(max(X_arr.max(), y_arr.max()))
span = (hi - lo) or 1.0

scaler_path = MODEL_DIR / "resale_scaler.json"
with open(scaler_path, "w") as fh:
    json.dump({"min": lo, "max": hi}, fh, indent=2)

print(f"Saved scaler bounds to: {scaler_path}")

# Scale training dataset
Xs = (X_arr - lo) / span
ys = (y_arr - lo) / span
Xs = Xs.reshape(Xs.shape[0], Xs.shape[1], 1)

print(f"Normalized dataset shape: {Xs.shape}")

In [ ]:
# 5. Compile & Train Recurrent Neural Network (LSTM)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(WINDOW, 1)),
    tf.keras.layers.LSTM(32, return_sequences=True),
    tf.keras.layers.LSTM(16),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1),
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

# Fit model
history = model.fit(
    Xs,
    ys,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# 6. Export Trained Model
model_path = MODEL_DIR / "resale_lstm.keras"
model.save(str(model_path))
print(f"Saved model successfully! -> {model_path}")